# Concatenate nuclei, membrane, and 2D shape descriptors
This notebook takes nuclei (including movement) feature dataframe, membrane feature dataframe, and 2D membrane slice dataframe (generated with 07_masterxml_dataframe.ipynb and 2d_slice_shape_descriptors.ipynb notebooks), and assumes upstream alignment has already been performed. It simply reads three CSVs and **concatenates columns (axis=1)**: nuclei, membrane, and 2D shape descriptors.

**Notes**
- No merges are performed here. Row order/index must already correspond across inputs.
- Optional strict alignment check is enabled by default.


In [ ]:
NUCLEI_CSV   = r'D:/Mari_Sixth_Dataset_Analysis/nuclei_membrane_tracking/nuclei_manual_dataset_withid.csv'
MEMBRANE_CSV = r'D:/Mari_Sixth_Dataset_Analysis/nuclei_membrane_tracking/membrane_manual_dataset_newannotations.csv'
SHAPE2D_CSV  = r'D:/Mari_Sixth_Dataset_Analysis/nuclei_membrane_tracking/membrane_manual_dataset_with2D.csv'
OUTPUT_CSV   = r'D:/Mari_Sixth_Dataset_Analysis/final_concatenated_features.csv'


## Imports & settings

In [ ]:
import os
import pandas as pd
from typing import Optional, Dict, List

# Optional column selection & dtype hints (leave None to use all columns)
NUC_COLUMNS = ['Unnamed: 0', 'ID', 'Spot source ID', 'Track ID_x', 'Track ID_y', 'Spot frame', 't', 't_hours', 'POSITION_X', 'POSITION_Y',
       'POSITION_Z', 'X_orig', 'Y_orig', 'Z_orig', 'Dividing', 'Number_Dividing', 'Radius', 'Eccentricity_Comp_First',
       'Eccentricity_Comp_Second', 'Eccentricity_Comp_Third',
       'Local_Cell_Density', 'Surface_Area', 'Speed', 'Motion_Angle_Z',
       'Motion_Angle_Y', 'Motion_Angle_X', 'Acceleration',
       'Distance_Cell_mask', 'Radial_Angle_Z', 'Radial_Angle_Y',
       'Radial_Angle_X', 'Cell_Axis_Z', 'Cell_Axis_Y', 'Cell_Axis_X', 'MSD',
       'TrackMate Track ID', 'Generation ID', 'Tracklet Number ID',
       'Track Duration', 'nuc_label',  'Spot track ID relabelled', 
       'Track ID numeric', 'cell_type', 
       'annotation']

MEM_COLUMNS = ['z', 'y', 'x', 'Radius',
       'Eccentricity_Comp_First', 'Eccentricity_Comp_Second',
       'Eccentricity_Comp_Third', 'Surface_Area',
       'Cell_Axis_Z', 'Cell_Axis_Y', 'Cell_Axis_X', 'Radial_Angle_Z', 'mem_label']

SHAPE2D_COLUMNS = ['Spot frame', 'nuc_label', 'mem_2d_area',
       'mem_2d_perimeter', 'mem_2d_eccentricity', 'mem_2d_solidity',
       'mem_2d_extent', 'mem_2d_axis_major_length', 'mem_2d_axis_minor_length',
       'mem_2d_feret_diameter_max']

FINAL_COLUMNS = ['Unnamed: 0','Track ID_x','Track ID_y','Spot frame','t','t_hours','POSITION_X','POSITION_Y','POSITION_Z','mem_x','mem_y','mem_z','X_orig','Y_orig','Z_orig',
 'Dividing','Number_Dividing','TrackMate Track ID','Generation ID','Tracklet Number ID','Track Duration','nuc_label','mem_label','Spot track ID relabelled','Track ID numeric',
 'cell_type','annotation','Local_Cell_Density','Speed','Motion_Angle_Z','Motion_Angle_Y','Motion_Angle_X','Acceleration','Distance_Cell_mask','Radial_Angle_Z','Radial_Angle_Y','Radial_Angle_X',
 'MSD','nuc_Radius','nuc_Eccentricity_Comp_First','nuc_Eccentricity_Comp_Second','nuc_Eccentricity_Comp_Third','nuc_Surface_Area','nuc_Cell_Axis_Z','nuc_Cell_Axis_Y','nuc_Cell_Axis_X',
 'mem_Radius','mem_Eccentricity_Comp_First','mem_Eccentricity_Comp_Second','mem_Eccentricity_Comp_Third','mem_Surface_Area','mem_Cell_Axis_Z','mem_Cell_Axis_Y','mem_Cell_Axis_X',
 'mem_Radial_Angle_Z','mem_2d_area','mem_2d_perimeter','mem_2d_eccentricity','mem_2d_solidity','mem_2d_extent','mem_2d_axis_major_length','mem_2d_axis_minor_length','mem_2d_feret_diameter_max']


# --- Column selections & dtype hints (optional, speeds parsing & reduces memory) ---
NUC_USECOLS: Optional[List[str]] = NUC_COLUMNS  # e.g., ['t','nuc_label','cell_id']
MEM_USECOLS: Optional[List[str]] = MEM_COLUMNS  # e.g., ['t','mem_label','cell_id']
SHAPE2D_USECOLS: Optional[List[str]] = SHAPE2D_COLUMNS  # e.g., ['t','mem_label','area','perimeter','circularity']

NUC_DTYPES: Dict[str, str] = {}
MEM_DTYPES: Dict[str, str] = {}
SHAPE_DTYPES: Dict[str, str] = {}

# Add suffixes to avoid duplicate column names
NUC_PREFIX   = 'nuc_'
MEM_PREFIX   = 'mem_'
SHAPE2D_PREFIX = ''

# Strict alignment check
STRICT_ALIGN = True


## Helpers

In [ ]:
def read_csv_typed(path: str, usecols=None, dtypes=None) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return pd.read_csv(path, usecols=usecols, dtype=dtypes, low_memory=False)

def add_suffix(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    df = df.copy()
    df.columns = [prefix + c for c in df.columns]
    return df

def check_alignment(dfa: pd.DataFrame, dfb: pd.DataFrame, dfc: pd.DataFrame) -> None:
    if len(dfa) != len(dfb) or len(dfa) != len(dfc):
        raise ValueError(f'Row count mismatch: nuclei={len(dfa)}, membrane={len(dfb)}, shape2D={len(dfc)}')
    if not dfa.index.equals(dfb.index) or not dfa.index.equals(dfc.index):
        raise ValueError('Index mismatch among inputs: ensure prior alignment before concatenation.')


## Read the three inputs

In [ ]:
nuc_df   = read_csv_typed(NUCLEI_CSV,   usecols=NUC_USECOLS,   dtypes=None) #or NUC_DTYPES
mem_df   = read_csv_typed(MEMBRANE_CSV, usecols=MEM_USECOLS,   dtypes=None)  #or MEM_DTYPES
shape_df = read_csv_typed(SHAPE2D_CSV,  usecols=SHAPE2D_USECOLS, dtypes=None)  #or SHAPE_DTYPES

if STRICT_ALIGN:
    check_alignment(nuc_df, mem_df, shape_df)

print('[nuc] rows:', len(nuc_df))
print('[mem] rows:', len(mem_df))
print('[shape2d] rows:', len(shape_df))


In [ ]:
nuc_df.rename(columns={'Radius' : 'nuc_Radius', 'Eccentricity_Comp_First' : 'nuc_Eccentricity_Comp_First',
       'Eccentricity_Comp_Second' : 'nuc_Eccentricity_Comp_Second', 'Eccentricity_Comp_Third' : 'nuc_Eccentricity_Comp_Third',
       'Surface_Area' : 'nuc_Surface_Area', 'Cell_Axis_Z' : 'nuc_Cell_Axis_Z', 'Cell_Axis_Y' : 'nuc_Cell_Axis_Y', 
       'Cell_Axis_X' : 'nuc_Cell_Axis_X'}, inplace=True)

mem_df.rename(columns={'z' : 'mem_z', 'y' : 'mem_y', 'x' : 'mem_x', 'Radius' : 'mem_Radius', 'Eccentricity_Comp_First' : 'mem_Eccentricity_Comp_First',
       'Eccentricity_Comp_Second' : 'mem_Eccentricity_Comp_Second', 'Eccentricity_Comp_Third' : 'mem_Eccentricity_Comp_Third',
       'Surface_Area' : 'mem_Surface_Area', 'Cell_Axis_Z' : 'mem_Cell_Axis_Z', 'Cell_Axis_Y' : 'mem_Cell_Axis_Y', 
       'Cell_Axis_X' : 'mem_Cell_Axis_X', 'Radial_Angle_Z' : 'mem_Radial_Angle_Z'}, inplace=True)

## Concatenate columns (axis=1)

In [ ]:
concatenated_df = pd.concat([nuc_df, mem_df], axis=1)
merged_df = pd.merge(concatenated_df, shape_df[['Spot frame', 'nuc_label', 'mem_2d_area',
       'mem_2d_perimeter', 'mem_2d_eccentricity', 'mem_2d_solidity',
       'mem_2d_extent', 'mem_2d_axis_major_length', 'mem_2d_axis_minor_length',
       'mem_2d_feret_diameter_max']], on=['Spot frame', 'nuc_label'], how='left')


## Save output

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
merged_df[FINAL_COLUMNS].to_csv(OUTPUT_CSV, index=False)
OUTPUT_CSV
